# EnKF Data Assimilation
Sequential FARSITE simulation with Ensemble Kalman Filter data assimilation.

This notebook runs FARSITE forward simulations and incorporates actual observations through EnKF at each timestep.

In [1]:
import sys
sys.path.insert(0, 'src')

import pandas as pd
import geopandas as gpd
import numpy as np
import json
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import requests
import time
from pyproj import Transformer
import contextily as ctx

from farsite import forward_pass_farsite, forward_pass_farsite_24h
from enkf import adjusted_state_EnKF_farsite
from geometry import geom_to_state, state_to_geom, align_states, plot_geometry
from config import OUTPUT_DIR, DATA_DIR, get_ensemble_size, DEFAULT_DIST_RES, DEFAULT_PERIM_RES, FIREMAP_WX_URL

from firemap import query_weather_for_timestep

## 1. Load Configuration and Data

In [2]:
# Load workflow configuration
with open(DATA_DIR / "workflow_config.json", 'r') as f:
    workflow_config = json.load(f)

FIRE_NAME = workflow_config["fire_name"]
LCP_PATH = workflow_config["lcp_path"]
IGNITION_DATE = pd.Timestamp(workflow_config["ignition_date"])
CONTAINMENT_DATE = pd.Timestamp(workflow_config["containment_date"])
N_PERIMETERS = workflow_config["n_perimeters"]
PERIMETERS_PATH = workflow_config["perimeters_path"]

SYNTHETIC = workflow_config["synthetic"]

print(f"Fire: {FIRE_NAME}")
print(f"Duration: {IGNITION_DATE.date()} to {CONTAINMENT_DATE.date()}")
print(f"Number of perimeter updates: {N_PERIMETERS}")
print(f"Landscape file: {LCP_PATH}")
print(f"geojson path: {PERIMETERS_PATH}")
print(f"Synthetic Fire?: {SYNTHETIC}")

Fire: SyntheticFire
Duration: 2020-01-01 to 2020-01-01
Number of perimeter updates: 25
Landscape file: /home/jovyan/work/WIFIRE-Digital-Twinners/wildfire-digital-twin/data/fire_perimeters/SyntheticFire.lcp
geojson path: /home/jovyan/work/WIFIRE-Digital-Twinners/wildfire-digital-twin/data/fire_perimeters/syntheticfire_perimeters.geojson
Synthetic Fire?: True


In [3]:
# Load perimeters — already sorted oldest→newest by fetch_fire_perimeters()
# perimeters_gdf = gpd.read_file(
#     OUTPUT_DIR / f"{FIRE_NAME.lower().replace(' ', '_')}_perimeters.geojson"
# )
perimeters_gdf = gpd.read_file(PERIMETERS_PATH)

perimeters_gdf['datetime'] = pd.to_datetime(perimeters_gdf['datetime'])

print(f"Loaded {len(perimeters_gdf)} perimeters")
for i, row in perimeters_gdf.iterrows():
    print(f"  [{i}] {row['datetime']}  —  {row.geometry.area/1e6:.2f} km²")

Loaded 25 perimeters
  [0] 2020-01-01 00:00:00  —  0.04 km²
  [1] 2020-01-01 00:30:00  —  0.16 km²
  [2] 2020-01-01 01:00:00  —  0.43 km²
  [3] 2020-01-01 01:30:00  —  0.70 km²
  [4] 2020-01-01 02:00:00  —  1.05 km²
  [5] 2020-01-01 02:30:00  —  1.40 km²
  [6] 2020-01-01 03:00:00  —  2.08 km²
  [7] 2020-01-01 03:30:00  —  2.68 km²
  [8] 2020-01-01 04:00:00  —  3.40 km²
  [9] 2020-01-01 04:30:00  —  4.53 km²
  [10] 2020-01-01 05:00:00  —  5.84 km²
  [11] 2020-01-01 05:30:00  —  8.04 km²
  [12] 2020-01-01 06:00:00  —  11.04 km²
  [13] 2020-01-01 06:30:00  —  13.93 km²
  [14] 2020-01-01 07:00:00  —  17.54 km²
  [15] 2020-01-01 07:30:00  —  21.69 km²
  [16] 2020-01-01 08:00:00  —  26.68 km²
  [17] 2020-01-01 08:30:00  —  30.84 km²
  [18] 2020-01-01 09:00:00  —  34.46 km²
  [19] 2020-01-01 09:30:00  —  37.06 km²
  [20] 2020-01-01 10:00:00  —  40.66 km²
  [21] 2020-01-01 10:30:00  —  43.09 km²
  [22] 2020-01-01 11:00:00  —  45.79 km²
  [23] 2020-01-01 11:30:00  —  48.04 km²
  [24] 2020-01-01

In [4]:
# Load weather location from config
wx_lat = workflow_config["weather_location"]["lat"]
wx_lon = workflow_config["weather_location"]["lon"]

print(f"\nWeather query location: {wx_lat:.4f}, {wx_lon:.4f}")
print("Weather will be queried dynamically for each timestep")


Weather query location: 32.5959, -116.8571
Weather will be queried dynamically for each timestep


## 2. Configure Simulation Parameters

In [5]:
# FARSITE parameters
DIST_RES = DEFAULT_DIST_RES  # meters
PERIM_RES = DEFAULT_PERIM_RES  # meters
FARSITE_TIMESTEP = pd.Timedelta(minutes=30)

# EnKF parameters
VSIZE = 500  # Observation noise (meters)
WSIZE = 200  # Model noise (meters)
RANDOM_SEED = 1234

# Initialize random number generator
rng = np.random.Generator(np.random.PCG64(RANDOM_SEED))

print("Simulation Parameters:")
print(f"  FARSITE distance resolution: {DIST_RES} m")
print(f"  FARSITE perimeter resolution: {PERIM_RES} m")
print(f"  FARSITE timestep: {FARSITE_TIMESTEP}")
print(f"  Observation noise (vsize): {VSIZE} m")
print(f"  Model noise (wsize): {WSIZE} m")

Simulation Parameters:
  FARSITE distance resolution: 150 m
  FARSITE perimeter resolution: 150 m
  FARSITE timestep: 0 days 00:30:00
  Observation noise (vsize): 500 m
  Model noise (wsize): 200 m


## 3. Initialize Tracking Variables

In [6]:
perimeters_gdf['geometry'].iloc[0].centroid.xy

(array('d', [-1931281.097517182]), array('d', [1270993.7641440213]))

In [7]:
# Storage for results
results = {
    'timestep': [],
    'datetime': [],
    'initial_geom': [],
    'observed_geom': [],
    'predicted_geom': [],
    'analysis_geom': [],
    'covariance': [],
    'n_vertices': [],
    'ensemble_size': [],
    'valid_ensemble_members': [],
    'mean_wind_speed': [],      # NEW: Track wind speed
    'mean_wind_direction': [],  # NEW: Track wind direction
    'n_weather_obs': []          # NEW: Number of weather observations
}

# Initialize state for first timestep
current_state = geom_to_state(perimeters_gdf['geometry'].iloc[0])
# n_vertex = current_state.shape[0] // 2
n_vertex = 20
X = 1000.0 * np.eye(2 * n_vertex)  # Initial covariance

print(f"Initial state:")
print(f"  Vertices: {n_vertex}")
print(f"  State dimension: {2 * n_vertex}")
print(f"  Covariance shape: {X.shape}")

Initial state:
  Vertices: 20
  State dimension: 40
  Covariance shape: (40, 40)


## 4. Sequential Simulation Loop

**FARSITE → EnKF Feedback Loop:**

Note: this loop can take 1+ hours to complete, depending on lengths and amounts of timesteps.

For each timestep (t → t+1):
1. **FARSITE Prediction**: Use current state (from previous EnKF analysis) as initial condition
2. **EnKF Assimilation**: Combine FARSITE prediction with actual observation
3. **Feedback**: EnKF analysis becomes the initial condition for next timestep

This creates a feedback loop where observations continuously correct the predictions.

#### Test FARSITE RUN file removals

In [8]:
# from farsite import Farsite

# poly = initial_geom
# new_params = farsite_params
# lcppath = LCP_PATH
# start_time = current_time

# farsite = Farsite(
#     poly, new_params,
#     start_time=start_time,
#     lcppath=lcppath
# )

In [9]:
# farsite.run()

In [10]:
# out = farsite.output_geom()

#### Main simulation loop

In [ ]:
# Main simulation loop
n_timesteps = len(perimeters_gdf) - 1  # We predict from t to t+1

print(f"Starting sequential simulation for {n_timesteps} timesteps...\n")
print("="*80)

for t in range(n_timesteps):
    print(f"\n{'='*80}")
    print(f"TIMESTEP {t} → {t+1}")
    print(f"{'='*80}")
    
    # Get current and next observation
    current_obs = perimeters_gdf.iloc[t]
    next_obs = perimeters_gdf.iloc[t + 1]
    
    current_time = pd.Timestamp(current_obs['datetime'])
    next_time = pd.Timestamp(next_obs['datetime'])
    dt = next_time - current_time
    
    print(f"Current time: {current_time}")
    print(f"Next time: {next_time}")
    print(f"Time delta: {dt}")
    
    # Geometries
    initial_geom = state_to_geom(current_state)
    observed_geom = next_obs['geometry']
    
    print(f"\nInitial perimeter area: {initial_geom.area / 1e6:.2f} km²")
    print(f"Observed perimeter area: {observed_geom.area / 1e6:.2f} km²")
    
    # ========================================================================
    # STEP 1: FARSITE Forward Prediction
    # ========================================================================
    print(f"\n[1/2] Running FARSITE forward prediction...")
    
    # Query weather for this specific timestep
    try:
        timestep_ws_list, timestep_wd_list = query_weather_for_timestep(
            lat=wx_lat,
            lon=wx_lon,
            start_time=current_time,
            end_time=next_time,
            verbose=True,
            synthetic=SYNTHETIC
        )
        mean_ws = np.mean(timestep_ws_list)
        mean_wd = np.mean(timestep_wd_list)

    except Exception as e:
        print(f"  Using fallback weather: {e}")
        mean_ws = 10.0
        mean_wd = 225.0
    
    # ======================================================
    # ========== 12h Prediction for comparison =============
    # ======================================================

    farsite_params = {
        'windspeed': int(mean_ws),
        'winddirection': int(mean_wd),
        'dt': pd.Timedelta(minutes=30)
    }
    
    predicted_geom = initial_geom
    for i in tqdm(range(24 - t), desc='Calculating 12h prediction...'):
        predicted_geom = forward_pass_farsite(
            poly=predicted_geom,
            params=farsite_params,
            start_time=current_time.strftime("%Y-%m-%d %H:%M:%S"),
            lcppath=LCP_PATH,
            dist_res=DIST_RES,
            perim_res=PERIM_RES
        )
    
    if predicted_geom is None:
        print(f"WARNING: FARSITE failed at timestep {t}. Skipping to next.")
        continue
    
    print(f"✓ FARSITE prediction complete")
    print(f"  Predicted perimeter area: {predicted_geom.area / 1e6:.2f} km²")
    
    # ========================================================================
    # STEP 2: EnKF Data Assimilation
    # ========================================================================
    print(f"\n[2/2] Running EnKF data assimilation...")
    
    # Prepare states
    observation_state = geom_to_state(observed_geom)
    
    # Align states to ensure consistent vertex count
    # n_vertex = n
    observation_state, current_state = align_states(
        [observation_state, current_state],
        vertex_count=n_vertex
    )
    
    # EnKF configuration
    n_states = 2 * n_vertex
    n_output = n_states
    n_samples = get_ensemble_size(n_vertex)
    
    print(f"  State dimension: {n_states}")
    print(f"  Ensemble size: {n_samples}")
    
    # Sample weather for ensemble from timestep-specific data
    if len(timestep_ws_list) > 1:
        idx = rng.integers(0, len(timestep_ws_list), size=n_samples)
        sampled_wslst = np.asarray(timestep_ws_list)[idx]
        sampled_wdlst = np.asarray(timestep_wd_list)[idx]
    else:
        # If only one observation, add noise around it
        sampled_wslst = mean_ws + rng.normal(0, 2.0, size=n_samples)  # +/- 2 mph
        sampled_wdlst = mean_wd + rng.normal(0, 15.0, size=n_samples)  # +/- 15 degrees
        sampled_wslst = np.maximum(sampled_wslst, 0)  # Wind speed can't be negative
        sampled_wdlst = np.fmod(sampled_wdlst + 360, 360)  # Keep in [0, 360)
    
    # Run EnKF
    try:
        adjusted_state, X, zkphat, xkhat, ykhat, xkphat = adjusted_state_EnKF_farsite(
            initial_state=current_state,
            observation_state=observation_state,
            observation_time=next_time,
            X=X,
            n_states=n_states,
            n_output=n_output,
            n_vertex=n_vertex,
            n_samples=n_samples,
            rng=rng,
            sampled_wslst=sampled_wslst,
            sampled_wdlst=sampled_wdlst,
            dt=FARSITE_TIMESTEP,
            vsize=VSIZE,
            wsize=WSIZE,
            dist_res=DIST_RES,
            perim_res=PERIM_RES,
            lcppath=LCP_PATH
        )
        
        analysis_geom = state_to_geom(adjusted_state)
        
        # Count valid ensemble members (non-zero forecasts)
        valid_members = np.sum(np.any(zkphat != 0, axis=0))
        
        print(f"✓ EnKF complete")
        print(f"  Valid ensemble members: {valid_members}/{n_samples}")
        print(f"  Analysis perimeter area: {analysis_geom.area / 1e6:.2f} km²")
        
    except Exception as e:
        print(f"ERROR in EnKF at timestep {t}: {e}")
        analysis_geom = observed_geom  # Fallback to observation
        adjusted_state = observation_state
        valid_members = 0
    
    # ========================================================================
    # STEP 3: Store Results
    # ========================================================================
    results['timestep'].append(t)
    results['datetime'].append(next_time)
    results['initial_geom'].append(initial_geom)
    results['observed_geom'].append(observed_geom)
    results['predicted_geom'].append(predicted_geom)
    results['analysis_geom'].append(analysis_geom)
    results['covariance'].append(X.copy())
    results['n_vertices'].append(n_vertex)
    results['ensemble_size'].append(n_samples)
    results['valid_ensemble_members'].append(valid_members)
    
    results['mean_wind_speed'].append(mean_ws)
    results['mean_wind_direction'].append(mean_wd)
    results['n_weather_obs'].append(len(timestep_ws_list))
    
        # Update state for next iteration (use analysis as new initial condition)
    current_state = adjusted_state
    
    print(f"\n✓ Timestep {t} → {t+1} complete")

print(f"\n{'='*80}")
print(f"SIMULATION COMPLETE")
print(f"{'='*80}")
print(f"\nProcessed {len(results['timestep'])} timesteps successfully")

Starting sequential simulation for 24 timesteps...


TIMESTEP 0 → 1
Current time: 2020-01-01 00:00:00
Next time: 2020-01-01 00:30:00
Time delta: 0 days 00:30:00

Initial perimeter area: 0.04 km²
Observed perimeter area: 0.16 km²

[1/2] Running FARSITE forward prediction...
  Querying weather: 2020-01-01 00:00:00 to 2020-01-01 00:30:00


Calculating 12h prediction...: 100%|██████████| 24/24 [02:04<00:00,  5.18s/it]


✓ FARSITE prediction complete
  Predicted perimeter area: 85.59 km²

[2/2] Running EnKF data assimilation...
  State dimension: 40
  Ensemble size: 300
Initial bounds: (-1931381.097517182, 1270893.7641440213, -1931181.097517182, 1271093.7641440213)
Observation bounds: (-1931433.0, 1270770.2327163722, -1930975.6895750621, 1271223.3433389661)


Running ensemble:   3%|▎         | 8/300 [00:29<14:53,  3.06s/it]

FARSITE output geometry is None


...

Running ensemble: 100%|██████████| 300/300 [15:26<00:00,  3.09s/it]


Valid samples: 300, Failed samples: 0
Calculating adjusted state (ensemble-space)
REACHED RETURN
✓ EnKF complete
  Valid ensemble members: 300/300
  Analysis perimeter area: 4.43 km²

✓ Timestep 5 → 6 complete

TIMESTEP 6 → 7
Current time: 2020-01-01 03:00:00
Next time: 2020-01-01 03:30:00
Time delta: 0 days 00:30:00

Initial perimeter area: 4.43 km²
Observed perimeter area: 2.68 km²

[1/2] Running FARSITE forward prediction...
  Querying weather: 2020-01-01 03:00:00 to 2020-01-01 03:30:00


Calculating 12h prediction...: 100%|██████████| 18/18 [01:32<00:00,  5.12s/it]


✓ FARSITE prediction complete
  Predicted perimeter area: 89.47 km²

[2/2] Running EnKF data assimilation...
  State dimension: 40
  Ensemble size: 300
Initial bounds: (-1933157.5039503507, 1270007.3082893062, -1930489.570908643, 1272090.2744166376)
Observation bounds: (-1931701.0, 1270163.3247939178, -1929521.5780050056, 1271924.6576715002)


Running ensemble:  18%|█▊        | 54/300 [03:52<18:09,  4.43s/it]

## 5. Visualize Results

In [ ]:
# Map visualization: Compare perimeters at each timestep
import contextily as ctx

n_timesteps = len(results['timestep'])
n_cols = 3
n_rows = (n_timesteps + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
axes = axes.flatten() if n_timesteps > 1 else [axes]

for idx, t in enumerate(results['timestep']):
    ax = axes[idx]
    
    # Create GeoDataFrames for plotting
    pred_gdf = gpd.GeoDataFrame([1], geometry=[results['predicted_geom'][idx]], crs="EPSG:5070")
    analysis_gdf = gpd.GeoDataFrame([1], geometry=[results['analysis_geom'][idx]], crs="EPSG:5070")
    obs_gdf = gpd.GeoDataFrame([1], geometry=[results['observed_geom'][idx]], crs="EPSG:5070")
    
    # Plot in order: FARSITE → EnKF → Observed
    pred_gdf.boundary.plot(ax=ax, color='blue', linewidth=4, linestyle='--', 
                          label='FARSITE Prediction', zorder=2, alpha=0.8)
    analysis_gdf.boundary.plot(ax=ax, color='red', linewidth=3, linestyle='-', 
                              label='EnKF Analysis', zorder=3, alpha=0.9)
    obs_gdf.boundary.plot(ax=ax, color='green', linewidth=2.5, linestyle=':', 
                         label='Observed (Truth)', zorder=4)
    
    # Add basemap with explicit zoom level
    try:
        ctx.add_basemap(
            ax=ax, 
            source=ctx.providers.OpenStreetMap.Mapnik, 
            crs="EPSG:5070",
            zoom=12,
            alpha=0.5,
            zorder=1
        )
    except Exception as e:
        print(f"Could not add basemap for timestep {t}: {e}")
    
    # Format
    ax.set_aspect('equal')
    
    # Get start and end dates
    start_date = perimeters_gdf.iloc[t]['datetime']
    end_date = results["datetime"][idx]
    time_diff = (end_date - start_date).total_seconds() / 3600
    
    # Calculate error statistics
    obs_area = results['observed_geom'][idx].area / 1e6
    pred_area = results['predicted_geom'][idx].area / 1e6
    enkf_area = results['analysis_geom'][idx].area / 1e6
    farsite_err = abs(pred_area - obs_area) / obs_area * 100
    enkf_err = abs(enkf_area - obs_area) / obs_area * 100
    
    ax.set_title(
        f'Day {t}→{t+1} ({time_diff:.0f}h)\n' +
        f'{start_date.strftime("%m/%d %H:%M")} → {end_date.strftime("%m/%d %H:%M")}\n' +
        f'FARSITE: {farsite_err:.1f}% | EnKF: {enkf_err:.1f}%',
        fontsize=9, weight='bold'
    )
    
    if idx == 0:
        ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
    
    ax.set_xlabel('Easting (m)', fontsize=9)
    ax.set_ylabel('Northing (m)', fontsize=9)
    ax.tick_params(labelsize=8)

# Hide unused subplots
for idx in range(n_timesteps, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle(f'{FIRE_NAME} - Perimeter Comparison (Daily Updates)\n' +
            'Blue=FARSITE Prediction | Red=EnKF Analysis | Green=Observed', 
            fontsize=15, weight='bold', y=0.998)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{FIRE_NAME.lower().replace(' ', '_')}_perimeter_maps.png", 
           dpi=200, bbox_inches='tight')
plt.show()

# Summary table
print(f"\n{'='*70}")
print(f"Fire Area and Error Summary")
print(f"{'='*70}")
print(f"Timestep | Observed | FARSITE  | EnKF     | FARSITE Err | EnKF Err")
print(f"{'─'*70}")
for idx, t in enumerate(results['timestep']):
    obs_area = results['observed_geom'][idx].area / 1e6
    pred_area = results['predicted_geom'][idx].area / 1e6
    enkf_area = results['analysis_geom'][idx].area / 1e6
    farsite_err = abs(pred_area - obs_area) / obs_area * 100
    enkf_err = abs(enkf_area - obs_area) / obs_area * 100
    print(f"  {t}→{t+1}    | {obs_area:6.2f}   | {pred_area:6.2f}   | {enkf_area:6.2f}   | "
          f"{farsite_err:6.1f}%     | {enkf_err:6.1f}%")
print(f"{'='*70}")

## 6. Save Results

In [ ]:
# Create results GeoDataFrame
results_gdf = gpd.GeoDataFrame({
    'timestep': results['timestep'],
    'datetime': results['datetime'],
    'type': ['analysis'] * len(results['timestep']),
    'geometry': results['analysis_geom'],
    'area_km2': [g.area / 1e6 for g in results['analysis_geom']],
    'n_vertices': results['n_vertices'],
    'ensemble_size': results['ensemble_size'],
    'valid_members': results['valid_ensemble_members']
}, crs="EPSG:5070")

# Save analysis results
output_path = OUTPUT_DIR / f"{FIRE_NAME.lower().replace(' ', '_')}_sequential_analysis.geojson"
results_gdf.to_file(output_path, driver="GeoJSON")
print(f"✓ Analysis results saved to {output_path}")

In [ ]:
# Save final state for potential continuation
from enkf import save_enkf_state

if len(results['timestep']) > 0:
    final_state_path = OUTPUT_DIR / f"{FIRE_NAME.lower().replace(' ', '_')}_final_state.npz"
    save_enkf_state(
        final_state_path,
        current_state,
        current_state,  # Use current as both initial and adjusted
        X,
        zkphat if 'zkphat' in locals() else np.array([]),
        xkhat if 'xkhat' in locals() else np.array([]),
        ykhat if 'ykhat' in locals() else np.array([]),
        xkphat if 'xkphat' in locals() else np.array([]),
        rng
    )
    print(f"✓ Final state saved to {final_state_path}")

In [ ]:
# Summary statistics
print("\n" + "="*80)
print("ANALYSIS SUMMARY")
print("="*80)
print(f"\nFire: {FIRE_NAME}")
print(f"Timesteps processed: {len(results['timestep'])}")
print(f"\nArea evolution:")
print(f"  Initial: {results['initial_geom'][0].area / 1e6:.2f} km²")
print(f"  Final (observed): {results['observed_geom'][-1].area / 1e6:.2f} km²")
print(f"  Final (FARSITE prediction): {results['predicted_geom'][-1].area / 1e6:.2f} km²")
print(f"  Final (EnKF analysis): {results['analysis_geom'][-1].area / 1e6:.2f} km²")
print(f"\nEnsemble performance:")
print(f"  Mean success rate: {np.mean(success_rate):.1f}%")
print(f"  Mean ensemble size: {np.mean(results['ensemble_size']):.0f}")
print(f"\nOutput files:")
print(f"  - {output_path.name}")
print(f"  - {final_state_path.name}")
print(f"  - Area evolution plot")
print(f"  - Spatial evolution plot")
print(f"  - Ensemble statistics plot")